In [1]:
import os
import cv2
import torch
import numpy as np

from groundingdino.util import box_ops
from groundingdino.models import build_model
import groundingdino.datasets.transforms as T
from groundingdino.util.slconfig import SLConfig
from groundingdino.util.utils import clean_state_dict
from groundingdino.util.inference import predict as dino_predict
from groundingdino.util.inference import annotate, load_image
from huggingface_hub import hf_hub_download


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
repo_id="ShilongLiu/GroundingDINO"
filename="groundingdino_swinb_cogcoor.pth"
config_filename="GroundingDINO_SwinB.cfg.py"
device='cpu'

cache_config_file = hf_hub_download(repo_id=repo_id, filename=config_filename)

args = SLConfig.fromfile(cache_config_file)
Dmodel = build_model(args)
args.device = device

cache_file = hf_hub_download(repo_id=repo_id, filename=filename)
checkpoint = torch.load(cache_file, map_location=device)
log = Dmodel.load_state_dict(clean_state_dict(checkpoint['model']), strict=False)

print("Check the latest .pth from: https://github.com/IDEA-Research/GroundingDINO/releases")

Dmodel.eval()

/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4382.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased
Check the latest .pth from: https://github.com/IDEA-Research/GroundingDINO/releases


GroundingDINO(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x DeformableTransformerEncoderLayer(
          (self_attn): MultiScaleDeformableAttention(
            (sampling_offsets): Linear(in_features=256, out_features=256, bias=True)
            (attention_weights): Linear(in_features=256, out_features=128, bias=True)
            (value_proj): Linear(in_features=256, out_features=256, bias=True)
            (output_proj): Linear(in_features=256, out_features=256, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout2): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (dropout3): Dropout(p=0.0, inplace=False)
          (norm2): LayerNorm((256,), eps=1e-05, elem

In [5]:
# Image transformation
image_source, image = load_image("/home/leo/MIE498Naturized/FullCycle/input/Frame_00000.tif")

boxes, logits, phrases = dino_predict(model=Dmodel,
                                      image=image,
                                      caption="Airfoil",
                                      box_threshold=0.3,
                                      text_threshold=0.3,
                                      device='cuda')
annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
cv2.namedWindow('Window', cv2.WINDOW_NORMAL)
# Set initial size
cv2.resizeWindow('Window', 800, 600)
cv2.imshow('Window', annotated_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()
# cv2.imwrite("/home/leo/MIE498Naturized/FullCycle/Thesis/Figures/GDOutSpace.png", annotated_frame)